### Weight regularization

In addition to the default cross-entropy loss, we experiment with two weight regularizers:

- **Ridge (L2):** penalizes the sum of squared weights.
- **Lasso (L1):** penalizes the sum of absolute weights.

Select the regularizer and its strength in the configuration cell below. The regularization penalty is added to the training loss only. Logged training and validation losses contain only cross-entropy, keeping the reported data loss comparable across runs.

In [ ]:
# Notebook configuration: replace command-line argument selection.
args.reg = "none"       # Choose: "none", "ridge", or "lasso"
args.reg_lambda = 1e-5  # Example strength; tune for your experiment

if args.reg not in {"none", "ridge", "lasso"}:
    raise ValueError("args.reg must be 'none', 'ridge', or 'lasso'.")

if args.reg_lambda < 0:
    raise ValueError("args.reg_lambda must be non-negative.")

In [ ]:
import torch
import torch.nn.functional as F
from contextlib import nullcontext
from tqdm.auto import tqdm


def runTraining(args):
    print(f">>> Setting up to train on {args.dataset} with {args.mode}")
    net, optimizer, device, train_loader, val_loader, K = setup(args)

    if args.mode == "full":
        # Supervise both background and foreground.
        loss_fn = CrossEntropy(idk=list(range(K)))
    elif args.mode == "partial" and args.dataset == "SEGTHOR":
        # Do not supervise the heart (class 2).
        loss_fn = CrossEntropy(idk=[0, 1, 3, 4])
    else:
        raise ValueError(
            f"Unsupported mode/dataset: {args.mode}/{args.dataset}"
        )

    # Loss: one value per batch. Dice: one value per sample and class.
    log_loss_tra = torch.zeros((args.epochs, len(train_loader)))
    log_dice_tra = torch.zeros(
        (args.epochs, len(train_loader.dataset), K)
    )
    log_loss_val = torch.zeros((args.epochs, len(val_loader)))
    log_dice_val = torch.zeros(
        (args.epochs, len(val_loader.dataset), K)
    )

    for e in range(args.epochs):
        for phase in ["train", "val"]:
            is_training = phase == "train"
            net.train(is_training)

            if is_training:
                loader = train_loader
                log_loss = log_loss_tra
                log_dice = log_dice_tra
                context = nullcontext
                desc = f"Training ({e + 1}/{args.epochs})"
            else:
                loader = val_loader
                log_loss = log_loss_val
                log_dice = log_dice_val
                context = torch.no_grad
                desc = f"Validation ({e + 1}/{args.epochs})"

            with context():
                j = 0
                progress = tqdm(
                    enumerate(loader),
                    total=len(loader),
                    desc=desc,
                    leave=False,
                )

                for i, data in progress:
                    img = data["images"].to(device)
                    gt = data["gts"].to(device)

                    if is_training:
                        optimizer.zero_grad()

                    assert 0 <= img.min().item() and img.max().item() <= 1
                    B = img.shape[0]

                    pred_logits = net(img)
                    pred_probs = F.softmax(pred_logits, dim=1)

                    # Metrics do not contribute to the gradient.
                    with torch.no_grad():
                        pred_seg = probs2one_hot(pred_probs)
                        batch_dice = dice_coef(pred_seg, gt)
                        log_dice[e, j:j + B, :] = batch_dice.detach().cpu()

                    loss = loss_fn(pred_probs, gt)

                    # Record cross-entropy only, excluding regularization.
                    log_loss[e, i] = loss.item()

                    if is_training:
                        total_loss = loss + regularization_penalty(
                            net, args.reg, args.reg_lambda
                        )
                        total_loss.backward()
                        optimizer.step()

                    # Advance the sample index for Dice logging.
                    j += B
                    progress.set_postfix(ce_loss=f"{loss.item():.4f}")

    return {
        "model": net,
        "optimizer": optimizer,
        "train_loss": log_loss_tra,
        "train_dice": log_dice_tra,
        "val_loss": log_loss_val,
        "val_dice": log_dice_val,
    }


# Run training and retain the model and logs for subsequent notebook cells.
results = runTraining(args)

However either method scores lower then the default

ridge: best val Dice = 0.7917 (epoch 22), final train loss = 0.0048

lasso: best val Dice = 0.7855 (epoch 24), final train loss = 0.0061